# Lezione 10. Apprendimento non supervisionato e rappresentazioni

**Corso: Advanced Deep Learning**

Questo quaderno è quello che esegui dal vivo durante la lezione. Alterna spiegazioni brevi con codice funzionante: ogni blocco teorico si appoggia alla cella di codice che si trova subito sotto.

## Obiettivi di apprendimento

Al termine di questa lezione, sai:

- Spiegare perché l'apprendimento non supervisionato è importante quando etichettare i dati ha un costo elevato
- Descrivere l'architettura di un autoencoder (codificatore più decodificatore) e la sua funzione di perdita
- Implementare e addestrare in PyTorch un autoencoder semplice sul dataset MNIST
- Confrontare visivamente le immagini originali con le loro ricostruzioni
- Esplorare lo spazio latente bidimensionale di un autoencoder e metterlo in relazione con le etichette reali delle cifre

## Il costo di etichettare i dati

Quasi tutto ciò che addestri nel deep learning richiede delle etichette: un'immagine e la sua classe, un testo e la sua traduzione. Etichettare costa caro. Classificare il milione di immagini di ImageNet ha richiesto migliaia di ore di lavoro umano, e in ambiti come la medicina l'etichettatura richiede inoltre personale specializzato.

Nel frattempo, il mondo produce una quantità quasi infinita di dati non etichettati: video, audio, testo, registrazioni di sensori. L'apprendimento non supervisionato cerca di sfruttare questi dati senza che qualcuno debba prima etichettarli.

## Comprimere per capire

Immagina di descrivere una fotografia in un messaggio molto breve: "una persona con giacca rossa, in piedi in un paesaggio innevato". Non invii i pixel, riassumi. Con quel riassunto, chi lo legge si fa un'idea fedele della foto, anche se non ricostruisce gli stessi pixel.

Un autoencoder (*autoencoder*) fa qualcosa di simile, in modo automatico e numerico. È composto da due parti:

- Un **codificatore**, che comprime l'input in un vettore molto più piccolo: la rappresentazione latente o codice
- Un **decodificatore**, che prende quel vettore e ricostruisce l'input originale a partire da esso

Il collo di bottiglia, cioè la dimensione ridotta del codice, impedisce alla rete di copiare l'input in modo banale: la costringe a scoprire quali informazioni meritano di essere conservate.

## Costruisci un autoencoder

Definisci il codificatore e il decodificatore con livelli lineari collegati da attivazioni ReLU (unità lineare rettificata). L'ultimo livello del decodificatore usa una sigmoide, perché i pixel di MNIST sono normalizzati tra zero e uno.

Addestri minimizzando l'errore di ricostruzione tra l'immagine originale e quella ricostruita. Nota che il ciclo di addestramento non usa mai l'etichetta della cifra.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

# Usa GPU si Colab te la ha asignado, si no, usa CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo de entrenamiento: {device}")

# --- Datos ---
# MNIST: dígitos de 28x28 en escala de grises, valores normalizados entre 0 y 1
transform = transforms.ToTensor()

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

DIM_ENTRADA = 28 * 28   # cada imagen se aplana a un vector de 784 valores
DIM_LATENTE = 32        # tamaño del cuello de botella: el "resumen" de cada imagen


# --- Definición del autocodificador ---
class Autocodificador(nn.Module):
    def __init__(self, dim_entrada=DIM_ENTRADA, dim_latente=DIM_LATENTE):
        super().__init__()
        # Codificador: comprime progresivamente hasta el cuello de botella
        self.codificador = nn.Sequential(
            nn.Linear(dim_entrada, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, dim_latente),
        )
        # Decodificador: expande de vuelta hasta el tamaño original
        # Sigmoid final porque los píxeles de MNIST están normalizados en [0, 1]
        self.decodificador = nn.Sequential(
            nn.Linear(dim_latente, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, dim_entrada),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.codificador(x)
        x_reconstruida = self.decodificador(z)
        return x_reconstruida, z


modelo = Autocodificador().to(device)
print(modelo)

# --- Función de pérdida y optimizador ---
criterio = nn.BCELoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=1e-3)


# --- Bucle de entrenamiento ---
def entrenar_una_epoca(modelo, loader, criterio, optimizador):
    modelo.train()
    perdida_total = 0.0
    for imagenes, _ in loader:  # las etiquetas no se usan: no hace falta supervisión
        imagenes = imagenes.view(imagenes.size(0), -1).to(device)  # aplana a vector

        optimizador.zero_grad()
        reconstruidas, _ = modelo(imagenes)
        perdida = criterio(reconstruidas, imagenes)
        perdida.backward()
        optimizador.step()

        perdida_total += perdida.item() * imagenes.size(0)
    return perdida_total / len(loader.dataset)


num_epocas = 10
for epoca in range(1, num_epocas + 1):
    perdida_train = entrenar_una_epoca(modelo, train_loader, criterio, optimizador)
    print(f"Época {epoca}/{num_epocas} | pérdida de reconstrucción: {perdida_train:.4f}")

## Originale vs. ricostruzione

Confronti dieci cifre di test con la loro ricostruzione. Aspettati ricostruzioni un po' sfocate, con bordi meno definiti: è la tipica impronta di una perdita basata sulla distanza tra pixel.

In [ ]:
modelo.eval()
imagenes_prueba, etiquetas_prueba = next(iter(test_loader))
imagenes_prueba = imagenes_prueba[:10]

with torch.no_grad():
    entrada_plana = imagenes_prueba.view(imagenes_prueba.size(0), -1).to(device)
    reconstruidas, _ = modelo(entrada_plana)
    reconstruidas = reconstruidas.view(-1, 28, 28).cpu()

fig, ejes = plt.subplots(2, 10, figsize=(15, 3))
for i in range(10):
    ejes[0, i].imshow(imagenes_prueba[i].squeeze(), cmap="gray")
    ejes[0, i].axis("off")
    ejes[1, i].imshow(reconstruidas[i], cmap="gray")
    ejes[1, i].axis("off")

ejes[0, 0].set_title("Original", loc="left", fontsize=10)
ejes[1, 0].set_title("Reconstruida", loc="left", fontsize=10)
plt.tight_layout()
plt.show()

## Esplora lo spazio latente

Il codice a trentadue dimensioni che hai usato prima ricostruisce bene, ma non puoi disegnarlo direttamente. Per vedere lo spazio latente con i tuoi occhi, addestri una seconda versione dello stesso autoencoder con un collo di bottiglia di sole due dimensioni.

Poi proietti le immagini di test in quello spazio bidimensionale e colori ogni punto in base alla cifra reale, anche se il modello non ha mai visto quell'etichetta durante l'addestramento.

In [ ]:
# Entrenas una segunda versión del autocodificador con cuello de botella de dos dimensiones
modelo_2d = Autocodificador(dim_latente=2).to(device)
optimizador_2d = torch.optim.Adam(modelo_2d.parameters(), lr=1e-3)

num_epocas_2d = 10
for epoca in range(1, num_epocas_2d + 1):
    perdida_train = entrenar_una_epoca(modelo_2d, train_loader, criterio, optimizador_2d)
    print(f"Época {epoca}/{num_epocas_2d} | pérdida de reconstrucción (2D): {perdida_train:.4f}")

# --- Proyección en el espacio latente ---
modelo_2d.eval()
imagenes_test, etiquetas_test = next(iter(test_loader))
imagenes_test = imagenes_test[:500]
etiquetas_test = etiquetas_test[:500]

with torch.no_grad():
    entrada_plana = imagenes_test.view(imagenes_test.size(0), -1).to(device)
    _, z = modelo_2d(entrada_plana)
    z = z.cpu().numpy()

plt.figure(figsize=(8, 6))
dispersion = plt.scatter(z[:, 0], z[:, 1], c=etiquetas_test.numpy(), cmap="tab10", s=10)
plt.colorbar(dispersion, label="Dígito real")
plt.xlabel("Dimensión latente 1")
plt.ylabel("Dimensión latente 2")
plt.title("Espacio latente de dos dimensiones")
plt.show()

## Riepilogo della lezione

- L'apprendimento non supervisionato sfrutta dati senza etichetta, evitando il costo dell'etichettatura manuale
- Un autoencoder impara comprimendo l'input in un codice latente (codificatore) e ricostruendolo a partire da quel codice (decodificatore)
- Il collo di bottiglia costringe la rete a privilegiare le informazioni più rilevanti e a scartare la ridondanza
- Addestri minimizzando l'errore di ricostruzione tra input e ricostruzione, senza usare alcuna etichetta
- Un collo di bottiglia a due dimensioni permette di visualizzare direttamente lo spazio latente, e le cifre uguali tendono ad aggregarsi anche se il modello non ha mai visto le loro etichette

**Apri ora il quaderno di pratica `../notebooks_practica/clase10_practica_aprendizaje_no_supervisado.ipynb` e segui gli esercizi proposti.**